# 실습 2-2 : 로지스틱 회귀

#### **<실습 내용>**

1. 실습 데이터 탐색 및 전처리
- 구조적 정보 / 통계적 정보
- 범주형 변수 처리 (One-Hot Encoding)
- 입출력 변수 분할
- 학습/테스트 데이터 분할
- 스케일링

2. 로지스틱 회귀
- 모델 학습 및 성능 평가
- 회귀 계수 해석

## 분석 준비

### 주요 라이브러리 호출

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

### 데이터 불러오기

In [ ]:
MF_Data = pd.read_csv(os.path.join(os.getcwd(), "dataset", "day2-2_data.csv"))

---

## 1) 데이터 탐색 및 전처리

### 1-1) 구조적 정보

In [ ]:
# 앞부분 데이터 확인
MF_Data.head(n=5)

In [ ]:
# 데이터 정보 확인
MF_Data.info()

In [ ]:
# 데이터 크기 확인
print("데이터 크기 :", MF_Data.shape)

- 현재 설비 상태(온도, 습도, 센서값, 이전 고장 후 경과시간)를 이용하여 향후 고장(Failure) 발생 여부를 예측하는 이진 분류 데이터셋

### 1-2) 통계적 정보

In [ ]:
# 요약 통계량 확인
MF_Data.describe(include="all")

### 1-3) 시각적 탐색

In [ ]:
# 히스토그램
MF_Data.hist(figsize=(30, 20))
plt.tight_layout()
plt.show()

> Measure2와 Measure3은 값이 소수의 정수값만 가지므로 **범주형 변수**로 처리해야 함

In [ ]:
plt.figure(figsize=(10,5))
plt.title("Failure Count")

counts = MF_Data["Failure"].value_counts()

sns.barplot(x=counts.index,
            y=counts.values,
            hue=counts.index)

plt.xticks(fontsize=20)
plt.xlabel("Value",fontsize=20)
plt.ylabel("Count",fontsize=15)
plt.show()

### 1-4) 입출력 변수 분할

In [ ]:
Y = MF_Data["Failure"]
X = MF_Data.drop(["Failure"], axis=1)

### 1-5) 범주형 변수 처리 (One-Hot Encoding)

> **One-Hot Encoding**은 범주형 변수의 각 항목을 0과 1로 표현하는 방법임
> - 범주가 k개인 변수를 k개의 새로운 이진 변수로 변환함
> - `pd.get_dummies()`를 사용하면 범주형 변수를 자동으로 One-Hot Encoding 할 수 있음

In [ ]:
# Measure2, Measure3을 범주형으로 변환
print("Measure2 고유값:", X["Measure2"].unique())
print("Measure3 고유값:", X["Measure3"].unique())

X["Measure2"] = X["Measure2"].astype("category")
X["Measure3"] = X["Measure3"].astype("category")

In [ ]:
# One-Hot Encoding 적용
X = pd.get_dummies(X)
data_columns = X.columns

X.head()

### 1-6) 출력변수 확인

> 불량(Yes) 비율이 약 0.9%로 **매우 불균형한 데이터**임

In [ ]:
# 클래스 비율 확인
print(Y.value_counts())
print()
print(Y.value_counts(normalize=True))

In [ ]:
# 출력변수 값 변경 (No -> 0, Yes -> 1)
Y = Y.replace({"No": 0, "Yes": 1}).infer_objects(copy=False)
Y.value_counts()

### 1-7) 학습/테스트 데이터 분할

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=0, stratify=Y)

print("전체 데이터 크기   :", X.shape)
print("학습 데이터 크기   :", X_train.shape)
print("테스트 데이터 크기 :", X_test.shape)

In [ ]:
# 클래스 비율이 유지되는지 확인
print("Y 클래스 비율")
print(np.round(Y.value_counts(normalize=True), 3))
print()
print("Y_train 클래스 비율")
print(np.round(Y_train.value_counts(normalize=True), 3))
print()
print("Y_test 클래스 비율")
print(np.round(Y_test.value_counts(normalize=True), 3))

### 1-8) 스케일링

> 로지스틱 회귀는 입력변수의 **스케일에 민감**하므로, MinMax 또는 Standard 스케일링을 적용해야함

In [ ]:
mc = MinMaxScaler()
X_train = mc.fit_transform(X_train)
X_test = mc.transform(X_test)

---

## 2) 로지스틱 회귀

> **로지스틱 회귀**는 출력변수가 범주형(정상/불량)인 경우에 사용되는 분류 모델임
> - 특정 사건이 발생할 **확률**을 예측하며, 확률이 기준값(보통 0.5) 이상이면 해당 클래스로 분류함
> - 로지스틱 함수: $p(X) = \frac{e^{\hat{\beta}_0 + \hat{\beta}_1 X_1 + \cdots + \hat{\beta}_k X_k}}{1 + e^{\hat{\beta}_0 + \hat{\beta}_1 X_1 + \cdots + \hat{\beta}_k X_k}}$
> - 회귀 계수 $\hat{\beta}_k$가 양수면 $X_k$가 증가할수록 불량 확률이 높아지고, 음수면 낮아짐

In [ ]:
# 분류 모형 성능 지표 산출 함수
def get_classscore(real, pred):
    print("Accuracy  : %.3f" % (accuracy_score(real, pred)))
    print("Precision : %.3f" % (precision_score(real, pred)))
    print("Recall    : %.3f" % (recall_score(real, pred)))
    print("F1-score  : %.3f" % (f1_score(real, pred)))
    print()
    print("혼동행렬")
    print(confusion_matrix(real, pred))

### 2-1) 모델 학습

In [ ]:
LR_model = LogisticRegression()
LR_model.fit(np.array(X_train), np.array(Y_train))

### 2-2) 회귀 계수 확인

> 로지스틱 회귀에서 회귀 계수의 절대값이 클수록 해당 변수의 **분류 영향력이 큼**.

In [ ]:
coef_data = pd.DataFrame({"Variable": data_columns, "Coef": np.abs(LR_model.coef_[0])})
coef_data.sort_values(by="Coef", ascending=False)

### 2-3) 성능 평가

> 분류 모델의 주요 성능 지표:
> - **Accuracy (정확도)**: 전체 중 올바르게 분류한 비율
> - **Precision (정밀도)**: 불량으로 예측한 것 중 실제 불량의 비율
> - **Recall (재현율)**: 실제 불량 중 불량으로 예측한 비율
> - **F1-score**: Precision과 Recall의 조화 평균

In [ ]:
LR_predict = LR_model.predict(X_test)
get_classscore(Y_test, LR_predict)

### 2-4) 확률 예측

> `predict_proba()`를 사용하면 각 클래스에 속할 **확률값**을 확인할 수 있음

In [ ]:
# 테스트 데이터에 대한 사건 발생 확률 예측
LR_predict_proba = LR_model.predict_proba(X_test)
LR_predict_proba[:5]

---

## 3) Vibe Coding 실습

### 3-1) 전처리 심화

**[과제 1]** 지수는 불량(Failure) 비율이 약 0.9%에 불과하다는 점을 확인했습니다. 이처럼 클래스 불균형이 심한 데이터에서는 전처리, 로지스틱 회귀 모델 학습, 평가 방법, 고도화 등을 어떻게 설정해야 하는지가 중요합니다. AI와 함께 이를 해결할 수 있는 방법을 찾아보고, 실제로 적용하여 모델 성능이 얼마나 개선되는지 확인해 보세요.

### 3-2) 모델링 및 고도화 심화

**[과제 2]** 지수는 predict_proba로 얻은 불량 확률을 그냥 0.5 기준으로 나누는 게 이 데이터에 적합한지 의문이 듭니다. 임계값(threshold)을 바꿔가며 Precision과 Recall이 어떻게 trade-off 되는지 살펴보는 방법을 AI와 논의하고, 이 데이터에 더 적합한 임계값을 찾아 적용해 보세요.

**[과제 3]** 지수는 선형 회귀에서 Ridge·Lasso를 활용해 모델의 복잡도를 조절할 수 있다는 것을 배웠습니다. 이에 따라 로지스틱 회귀도 과적합을 방지하거나 모델의 복잡도를 조절할 수 있는 방법이 있는지 궁금해졌습니다. AI와 함께 모델 복잡도를 조절하는 기법을 찾아보고, 다양한 설정을 적용하여 성능과 회귀 계수의 변화를 비교해 보세요.

**[과제 4]** 지수는 모델이 구체적으로 어떤 케이스에서 틀리는지 직접 눈으로 확인하고 싶습니다. 모델이 잘못 예측한 데이터를 추려내는 방법을 AI와 상의해서 추출하고, 이 데이터들이 다른 데이터와 어떤 특징 차이를 보이는지 살펴보세요.